# 02 · Phase 0 known result, leak test, step budget  (~2–3 GPU-h)

**Stage 1: E0.6.** A1 and A2 at (N=64, F=512, P=64, a), 6,000 steps, with val recall every 250 steps
(~30 min). MQAR has a long plateau (recall stuck at 1/N) before the model suddenly learns the key→value
binding. This curve shows when that happens, and gives the real per-run time.

**Stage 2, with the step count from stage 1** (pre-registered rule: A1's first step ≥ 98% of its max val
recall, clamped to [2000, 4000]; or set `E04_STEPS` yourself):
* **E0.4 (test 7):** vanilla linear attention (LA) vs GDN (A1) on plain MQAR, N ∈ {16, 32, 64, 128, 256}, 3 seeds; T1 at 1 seed.
* **E0.5 (test 6 at scale):** the Engram-only probe A6 at every N; it must be at chance.

The budget line before stage 2 prints the E0.4 GPU-hours at that step count.

**Outputs:** test-7 table and verdict, N\* proposal, step-budget proposal, and the Phase 1 GPU-hour estimate from
measured throughput.

**Stop if test 7 fails** (design §6: the pipeline is wrong). **Then you:** fill `configs/prereg.yaml` (N_star,
phase1_steps) and the PREREG.md "Phase 0 outcomes" table, commit, push, and pin that commit as `REPO_REF` in
every Phase 1 notebook.

In [ ]:
# ===================== SETTINGS: edit this cell, then "Run All" (or "Save Version -> Save & Run All") =====================
REPO_URL     = "https://github.com/Shiveshrane/Engram.git"   # None -> copy the code from CODE_DATASET instead
REPO_REF     = "main"           # pin the PREREG commit hash for Phase 1 / 2 runs
CODE_DATASET = "/kaggle/input/engramdelta-src"   # Kaggle Dataset with the repo (used when REPO_URL is None)
HF_REPO      = None             # e.g. "your-hf-user/engramdelta-runs" (private dataset repo; needs Kaggle Secret HF_TOKEN)
GDN_BACKEND  = "torch"          # "torch" (pure PyTorch, always works) or "fla" (only if NB00 says PASS)
FLA_SPEC     = "fla-core==0.5.2"                 # FLA's kernel package (fla.ops); installed with --no-deps
PREV_OUTPUTS = []               # outputs of earlier sessions attached as inputs, e.g. ["/kaggle/input/engramdelta-11-phase1-h1-grid"]
NCCL_ENV     = {}               # set to {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"} if NB00's first NCCL test hung

In [ ]:
# ===================== Session setup (no edits needed) =====================
import os, sys, subprocess, shutil, pathlib, json, glob, time
NEEDS_GPU, INSTALL_FLA = True, False
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
# Locally (notebook opened from the repo's notebooks/ folder) the repo is the parent directory.
REPO = WORK / "engramdelta" if ON_KAGGLE else (WORK.parent if WORK.name == "notebooks" else WORK)

def run(cmd, check=True, env=None, timeout=None, show=None):
    """Run a shell command in the repo and stream its output into this notebook."""
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, cwd=str(REPO) if REPO.exists() else str(WORK))
    t0 = time.time()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if timeout and time.time() - t0 > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        p.kill(); print(f"!! timed out after {timeout}s"); rc = -9
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {show or cmd}")
    return rc

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

# 1. code
if not REPO.exists():
    if REPO_URL:
        url, tok = REPO_URL, secret("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://{tok}@")
        run(f"git clone -q {url} {REPO}", show=f"git clone {REPO_URL}")
        run(f"git -C {REPO} checkout -q {REPO_REF}")
    else:
        shutil.copytree(CODE_DATASET, REPO)
os.chdir(REPO); sys.path.insert(0, str(REPO))
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    commit = open("GIT_COMMIT").read().strip() if os.path.exists("GIT_COMMIT") else "unknown"
os.environ["GIT_COMMIT"] = commit
print("code commit:", commit)

# 2. packages (never let pip replace Kaggle's torch)
run("pip install -q --no-deps einops sympy")
if INSTALL_FLA or GDN_BACKEND == "fla":
    run(f"pip install -q --no-deps {FLA_SPEC}", check=False)

# 3. environment for every launched run
os.environ["ENGRAMDELTA_GDN_BACKEND"] = GDN_BACKEND
os.environ["ENGRAMDELTA_CKPT_DIR"] = str(WORK / "ckpt")
os.environ.update(NCCL_ENV)
if HF_REPO and secret("HF_TOKEN"):
    os.environ["HF_REPO"], os.environ["HF_TOKEN"] = HF_REPO, secret("HF_TOKEN")
    print("HF Hub sync: ON ->", HF_REPO)
else:
    print("HF Hub sync: OFF (results persist via this notebook's output only)")

# 4. restore results / checkpoints from earlier sessions (never overwrites existing files)
def restore(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n
for prev in PREV_OUTPUTS:
    n1 = restore(os.path.join(prev, "engramdelta", "results"), str(REPO / "results"))
    n2 = restore(os.path.join(prev, "ckpt"), str(WORK / "ckpt"))
    print(f"restored from {prev}: {n1} result files, {n2} checkpoint files")
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull results", check=False)

# 5. hardware
if NEEDS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
NPROC = max(1, torch.cuda.device_count())
if NEEDS_GPU and NPROC < 2:
    print("!! WARNING: fewer than 2 GPUs. Set Accelerator = 'GPU T4 x2' in the notebook settings.")

## Stage 1 · E0.6: training curve and real per-run time

In [ ]:
GROUPS = ['E0.6']     # run in this order (later groups may need earlier results, e.g. LRs from E1.0a)
SESSION_HOURS = 1.0   # leave margin below Kaggle's 12 h session limit
EXTRA = ""
t_start = time.time()
for g in GROUPS:
    left = SESSION_HOURS - (time.time() - t_start) / 3600
    if left < 0.15:
        print("session budget used up; rerun this notebook to continue"); break
    run(f"python -m engramdelta.scripts.make_manifest --group {g} --out manifests {EXTRA}")
    run(f"python -m engramdelta.scripts.budget manifests/{g}.yaml --nproc {NPROC}", check=False)
    rc = run(f"python -m engramdelta.scripts.sweep manifests/{g}.yaml --nproc {NPROC} --session-hours {left:.2f}", check=False)
    if rc != 0:
        print(f"!! sweep for {g} reported failures or stopped early (exit {rc}). Check results/_failures before continuing.")

In [ ]:
from engramdelta import analysis
E04_STEPS = "auto"      # "auto" = pre-registered rule below, or an int (e.g. 3000) to override
sb = analysis.step_budget_proposal("results")
for arm, v in sb.items():
    print(f"{arm}: max val recall {v['max_val_recall']:.3f}, first step >= 98% of max: {v['first_step_at_98pct']} "
          f"-> proposal {v['proposal']} | {v['tok_per_s']:.0f} tok/s, {v['train_seconds'] / 60:.1f} min for 6000 steps")
    print("   curve (step, val recall):", [(s, round(r, 3)) for s, r in v["curve"]])
if E04_STEPS == "auto":
    E04_STEPS = int(min(4000, max(2000, sb["A1"]["proposal"])))
if sb["A1"]["max_val_recall"] < 0.5:
    print("!! A1 stayed below 50% val recall for all 6000 steps at N=64, F=512. Look at the curve; E0.4 at small N may still"
          " be fine, but consider a larger E04_STEPS before spending the E0.4 budget.")
print("\n>> E0.4 / E0.5 will train for", E04_STEPS, "steps")

## Stage 2 · E0.4 (test 7) and E0.5 (leak test)

In [ ]:
GROUPS = ['E0.4', 'E0.5']     # run in this order (later groups may need earlier results, e.g. LRs from E1.0a)
SESSION_HOURS = 2.5   # leave margin below Kaggle's 12 h session limit
EXTRA = f"--extra-set train.steps={E04_STEPS}"
t_start = time.time()
for g in GROUPS:
    left = SESSION_HOURS - (time.time() - t_start) / 3600
    if left < 0.15:
        print("session budget used up; rerun this notebook to continue"); break
    run(f"python -m engramdelta.scripts.make_manifest --group {g} --out manifests {EXTRA}")
    run(f"python -m engramdelta.scripts.budget manifests/{g}.yaml --nproc {NPROC}", check=False)
    rc = run(f"python -m engramdelta.scripts.sweep manifests/{g}.yaml --nproc {NPROC} --session-hours {left:.2f}", check=False)
    if rc != 0:
        print(f"!! sweep for {g} reported failures or stopped early (exit {rc}). Check results/_failures before continuing.")

## Test 7: known result (GDN vs vanilla linear attention)

In [ ]:
from engramdelta import analysis
df = analysis.load("results")
chk = analysis.known_result_check(df)
print(chk["table"][["N", "mean_A", "mean_B", "pooled_std", "diff_A_minus_B", "exceeds_2sigma", "T1_recall"]]
      .rename(columns={"mean_A": "A1", "mean_B": "LA"}).to_string(index=False))
print("\ncriterion 1 (A1 > LA + 2 sigma at >= 2 N):", chk["criterion_1_pass"], f"({chk['n_wins']} N)")
print("criterion 2 (A1 non-increasing beyond the knee):", chk["criterion_2_pass"])
print("TEST 7:", "PASS" if chk["pass"] else "FAIL -> STOP, the pipeline is wrong (design §6)")
print("\nN* proposal (largest N with A1 mean recall >= 0.9):", chk["N_star"])

## Test 6 at scale: A6 leak check

In [ ]:
print(analysis.leak_table(df).to_string(index=False))
run("python -m pytest tests/test_repro_known.py -q -s -rs", check=False)

## Step budget (E0.6) and measured run time

In [ ]:
sb = analysis.step_budget_proposal("results")
for arm, v in sb.items():
    print(f"{arm}: max val recall {v['max_val_recall']:.3f}, first step >= 98% of max: {v['first_step_at_98pct']}, "
          f"proposal {v['proposal']} steps | {v['tok_per_s']:.0f} tok/s, {v['train_seconds']/60:.1f} min for 6000 steps")
run("python -m engramdelta.scripts.make_plots --results-dir results --out figures", check=False)
from IPython.display import Image, display
for f in sorted(glob.glob("figures/E0.*.png")):
    display(Image(f))

## Proposed pre-registered values and the Phase 1 budget (from measured throughput)

In [ ]:
import yaml
pr = yaml.safe_load(open("configs/prereg.yaml"))
prop = dict(pr, N_star=chk["N_star"], phase1_steps=max(v["proposal"] for v in sb.values()) if sb else None)
yaml.safe_dump(prop, open("configs/prereg.proposed.yaml", "w"), sort_keys=False)
print(open("configs/prereg.proposed.yaml").read())
# Budget: LRs are not known yet, so use placeholder LRs (they do not change the run time)
tmp = dict(prop, lr={"A1": 1e-3, "A2": 1e-3, "T1": 1e-3})
yaml.safe_dump(tmp, open("/tmp/prereg_budget.yaml", "w"))
run("python -m engramdelta.scripts.make_manifest --group all-phase1 --prereg /tmp/prereg_budget.yaml --out /tmp/manifests_p1")
run(f"python -m engramdelta.scripts.budget '/tmp/manifests_p1/*.yaml' --nproc {NPROC} --quota-hours 13", check=False)

## Your turn (no GPU needed)
1. Check the test-7 verdict. If it FAILs, stop here.
2. Copy `N_star` and `phase1_steps` from `configs/prereg.proposed.yaml` into `configs/prereg.yaml`. If the budget above
   says OVER BUDGET, decide on cuts (plan §6.5) **before** committing.
3. Fill the "Phase 0 outcomes" table in PREREG.md, commit, push, and put the commit hash in PREREG.md and as `REPO_REF`
   in notebooks 10–14.

In [ ]:
# ===================== End of session: persist =====================
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync push results", check=False)
n = sum(len(f) for _, _, f in os.walk(REPO / "results"))
print(f"{n} files under results/. They are part of this notebook's output: Save Version keeps them.")
print("Next session: attach this notebook's output as an input and add its path to PREV_OUTPUTS (or use HF_REPO).")